# GTAN-alone, uid-disjoint — kNN similarity + identity edges, reference uids, 11-dim edge_attr

Approach 2: KMeans removed. Similarity edges = GPU batched cosine top-k (causal, cross-uid). Identity edges kept (uid, card1_addr1, card1_addr1_P_emaildomain, DeviceInfo). 40% of TRAIN uids are reference (labels revealed); query train uids (60%) are supervised; val scored. edge_attr = [rec1,rec7,rec30, is_uid,is_c1a1,is_c1a1e,is_dev,is_sim,is_self, rel_count, cosine]. GTAN alone — no CNN/LSTM. **Run All.**

In [12]:
!pip install -q torch_geometric

In [13]:
import torch
TORCH=torch.__version__
!pip install -q pyg-lib torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{TORCH}.html
# optional FAISS backend (set SIM_BACKEND='faiss'): 
# !pip install -q faiss-cpu

In [14]:
import torch_scatter, torch_sparse
print('GNN backend ready')

GNN backend ready


In [15]:
# ===== Config =====
import os, gc, math, time, copy, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch_geometric.nn import TransformerConv
from torch_geometric.utils import add_remaining_self_loops
from torch_geometric.loader import NeighborLoader
from torch_geometric.data import Data
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
OUT = Path("/kaggle/working"); OUT.mkdir(parents=True, exist_ok=True)

# ===== MODES =====
USE_REFERENCE_NODES    = False    # True: 40% of TRAIN uids revealed as references (approach 2)
                                 # False: ALL train labels masked (no reveal) -> pure feature+structure GNN
ADD_TIME_COUNT_FEATURES = False  # True: add uid per-day / per-week / per-month transaction counts as node features

SEED=42; FRAC_TRAIN=0.8; REF_FRAC=0.40; REF_SEED=123
IDENTITY_COLS=["uid","card1_addr1","card1_addr1_P_emaildomain","DeviceInfo"]
EDGE_PER_TRANS=12; K_SIM=15
DECAY_TAUS=(1.0,7.0,30.0)
USE_COSINE=True
SIM_BACKEND="gpu"; SIM_BATCH=1024
GTAN_HIDDEN=32; GTAN_HEADS=4; GTAN_LAYERS=2; GTAN_DROP=0.2
GTAN_EPOCHS=30; GTAN_LR=3e-4; GTAN_WD=1e-5; EARLY_STOP_PATIENCE=4
GTAN_BATCH=4096; GTAN_NEIGH=(15,10); N_SEEDS=3
torch.manual_seed(SEED); np.random.seed(SEED)


device: cuda


In [16]:
# ===== Load data (copy4 has features + identity + TransactionDT) =====
df = pd.read_parquet("/kaggle/input/datasets/bachhoviet/parquets/X_train_copy4.parquet")
df.index = df.index.astype(np.int64)
y_all = pd.read_parquet("/kaggle/input/datasets/bachhoviet/parquets/y_train.parquet")["isFraud"]
y_all = y_all.reindex(df.index).to_numpy().astype(np.int64)

BASE_FEAT = [c for c in ['TransactionAmt', 'ProductCD_FE', 'card1', 'card2', 'card3', 'card5', 'card6_FE', 'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain_FE', 'R_emaildomain_FE', 'C1', 'C2', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13', 'C14', 'D1', 'D2', 'D3', 'D4', 'D5', 'D10', 'D11', 'D15', 'M1', 'M2', 'M3', 'M4_FE', 'M6', 'M7', 'M8', 'M9', 'V1', 'V3', 'V4', 'V6', 'V8', 'V11', 'V13', 'V14', 'V17', 'V20', 'V23', 'V26', 'V27', 'V30', 'V36', 'V37', 'V40', 'V41', 'V44', 'V47', 'V48', 'V54', 'V56', 'V59', 'V62', 'V65', 'V67', 'V68', 'V70', 'V76', 'V78', 'V80', 'V82', 'V86', 'V88', 'V89', 'V91', 'V107', 'V108', 'V111', 'V115', 'V117', 'V120', 'V121', 'V123', 'V124', 'V127', 'V129', 'V130', 'V136', 'V138', 'V139', 'V142', 'V147', 'V156', 'V160', 'V162', 'V165', 'V166', 'V169', 'V171', 'V173', 'V175', 'V176', 'V178', 'V180', 'V182', 'V185', 'V187', 'V188', 'V198', 'V203', 'V205', 'V207', 'V209', 'V210', 'V215', 'V218', 'V220', 'V221', 'V223', 'V224', 'V226', 'V228', 'V229', 'V234', 'V235', 'V238', 'V240', 'V250', 'V252', 'V253', 'V257', 'V258', 'V260', 'V261', 'V264', 'V266', 'V267', 'V271', 'V274', 'V277', 'V281', 'V283', 'V284', 'V285', 'V286', 'V289', 'V291', 'V294', 'V296', 'V297', 'V301', 'V303', 'V305', 'V307', 'V309', 'V310', 'V314', 'V320', 'id_01', 'id_02', 'id_03', 'id_04', 'id_05', 'id_06', 'id_09', 'id_10', 'id_11', 'id_12', 'id_13', 'id_15_FE', 'id_16', 'id_17', 'id_18', 'id_19', 'id_20', 'id_28', 'id_29', 'id_31_FE', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo_FE', 'cents', 'dollars', 'addr1_FE', 'card1_FE', 'card2_FE', 'card3_FE', 'card1_addr1', 'card1_addr1_P_emaildomain', 'card1_addr1_FE', 'card1_addr1_P_emaildomain_FE', 'TransactionAmt_card1_mean', 'TransactionAmt_card1_std', 'TransactionAmt_card1_addr1_mean', 'TransactionAmt_card1_addr1_std', 'TransactionAmt_card1_addr1_P_emaildomain_mean', 'TransactionAmt_card1_addr1_P_emaildomain_std', 'D9_card1_mean', 'D9_card1_std', 'D9_card1_addr1_mean', 'D9_card1_addr1_std', 'D9_card1_addr1_P_emaildomain_mean', 'D9_card1_addr1_P_emaildomain_std', 'D11_card1_mean', 'D11_card1_std', 'D11_card1_addr1_mean', 'D11_card1_addr1_std', 'D11_card1_addr1_P_emaildomain_mean', 'D11_card1_addr1_P_emaildomain_std', 'is_december', 'is_holiday', 'uid_FE', 'delta_seconds_prev', 'uid_count_so_far', 'uid_prev_amt', 'uid_amt_diff_prev', 'uid_amt_ratio_prev', 'uid_amt_cummean', 'uid_amt_cummax', 'DT_hour_sin', 'DT_hour_cos', 'DT_day_week_sin', 'DT_day_week_cos', 'DT_day_month_sin', 'DT_day_month_cos', 'DT_week_month_sin', 'DT_week_month_cos'] if c in df.columns]

EXTRA_FEAT = []
if ADD_TIME_COUNT_FEATURES:
    # number of transactions in that day / week / month for that uid
    day  = (df["TransactionDT"].to_numpy() // 86400).astype(np.int64)
    tmp  = pd.DataFrame({"uid": df["uid"].to_numpy(), "day": day, "week": day//7, "month": day//30})
    df["uid_day_ct"]   = tmp.groupby(["uid","day"]).transform("size").to_numpy()
    df["uid_week_ct"]  = tmp.groupby(["uid","week"]).transform("size").to_numpy()
    df["uid_month_ct"] = tmp.groupby(["uid","month"]).transform("size").to_numpy()
    EXTRA_FEAT = ["uid_day_ct","uid_week_ct","uid_month_ct"]
    print("added time-count features:", EXTRA_FEAT)

FEAT = BASE_FEAT + EXTRA_FEAT
print("node features:", len(FEAT), "| rows:", len(df))
X_raw   = df[FEAT].apply(pd.to_numeric, errors="coerce").fillna(-1).to_numpy().astype(np.float32)
times   = df["TransactionDT"].to_numpy().astype(np.float64)
uids    = df["uid"].to_numpy()
id_vals = {c: df[c].to_numpy() for c in IDENTITY_COLS}
N = len(df)


node features: 212 | rows: 590540


In [17]:
# ===== uid-disjoint split + (optional) reference uids =====
uid_counts = df["uid"].value_counts(dropna=False)
uids_arr = uid_counts.index.to_numpy(); counts_arr = uid_counts.values.astype(np.int64)
rng = np.random.default_rng(SEED); perm = rng.permutation(len(uids_arr))
uids_shuf = uids_arr[perm]; counts_shuf = counts_arr[perm]
cut = int(np.searchsorted(np.cumsum(counts_shuf), int(FRAC_TRAIN*counts_shuf.sum()))) + 1
train_uids = set(uids_shuf[:cut].tolist()); val_uids = set(uids_shuf[cut:].tolist())

u = df["uid"].to_numpy()
train_mask = np.isin(u, list(train_uids)); val_mask = ~train_mask

if USE_REFERENCE_NODES:
    tr_uid_arr = np.array(sorted(train_uids))
    ref_rng = np.random.default_rng(REF_SEED)
    n_ref = int(REF_FRAC*len(tr_uid_arr))
    ref_uids = set(ref_rng.permutation(tr_uid_arr)[:n_ref].tolist())
    np.save(OUT/"reference_uids.npy", np.array(sorted(ref_uids)))
    ref_mask   = np.isin(u, list(ref_uids))      # revealed
    query_mask = train_mask & ~ref_mask          # masked + supervised
else:
    ref_mask   = np.zeros(N, dtype=bool)         # NO reveal: every train node masked
    query_mask = train_mask                      # supervise on all train

ref_idx   = np.flatnonzero(ref_mask)
query_idx = np.flatnonzero(query_mask)
val_idx   = np.flatnonzero(val_mask)
print(f"MODE: reference_nodes={USE_REFERENCE_NODES} | time_count_feats={ADD_TIME_COUNT_FEATURES}")
print(f"train rows={train_mask.sum():,} (ref={ref_mask.sum():,} query={query_mask.sum():,}) | val rows={val_mask.sum():,}")
print(f"fraud: query={y_all[query_idx].mean():.4f} val={y_all[val_idx].mean():.4f}" +
      (f" ref={y_all[ref_idx].mean():.4f}" if len(ref_idx) else " (no reference reveal)"))


MODE: reference_nodes=False | time_count_feats=False
train rows=472,434 (ref=0 query=472,434) | val rows=118,106
fraud: query=0.0346 val=0.0365 (no reference reveal)


In [18]:
# ===== Scale (fit on TRAIN) + L2-normalize for cosine =====
sc = StandardScaler().fit(X_raw[train_mask])
X_scaled = sc.transform(X_raw).astype(np.float32)       # GTAN node features
norm = np.linalg.norm(X_scaled, axis=1, keepdims=True); norm[norm==0]=1.0
X_norm = (X_scaled / norm).astype(np.float32)           # for cosine similarity
print("X_scaled", X_scaled.shape, "| X_norm", X_norm.shape)


X_scaled (590540, 212) | X_norm (590540, 212)


In [19]:
# ===== kNN similarity edges: GPU batched cosine top-k (causal, cross-uid) =====
@torch.no_grad()
def knn_gpu(Xn_np, times, uids, k, dev, batch=1024):
    N = Xn_np.shape[0]
    Xn = torch.from_numpy(Xn_np).to(dev)
    t  = torch.tensor(times, device=dev)
    uu = torch.tensor(pd.factorize(uids)[0], device=dev)   # int codes for uid
    src_l, dst_l = [], []
    for s0 in range(0, N, batch):
        s1 = min(s0+batch, N)
        sims = Xn[s0:s1] @ Xn.T                              # (B,N) cosine (normalized)
        qt = t[s0:s1].unsqueeze(1); qu = uu[s0:s1].unsqueeze(1)
        valid = (t.unsqueeze(0) < qt) & (uu.unsqueeze(0) != qu)   # earlier time AND different uid
        sims = sims.masked_fill(~valid, float("-inf"))
        kk = min(k, N)
        topv, topi = torch.topk(sims, kk, dim=1)
        m = torch.isfinite(topv)
        rows = (torch.arange(s1-s0, device=dev).unsqueeze(1).expand(-1, kk)[m] + s0)
        cols = topi[m]
        dst_l.append(rows.cpu()); src_l.append(cols.cpu())   # src=similar PAST -> dst=current
        del sims, valid, topv, topi
    if device.type=="cuda": torch.cuda.empty_cache()
    if not src_l: z=np.array([],np.int64); return z,z
    return torch.cat(src_l).numpy(), torch.cat(dst_l).numpy()

def knn_faiss(Xn_np, k):   # cosine via inner product on L2-normalized (no causal filter -> over-query then filter)
    import faiss
    index = faiss.IndexFlatIP(Xn_np.shape[1]); index.add(Xn_np)
    _, I = index.search(Xn_np, k+1)   # +1 to drop self
    return I


In [20]:
# ===== Build combined edge_index + 11-dim edge_attr (identity + similarity, coalesced) =====
def _valid_id(v):
    v=np.asarray(v); ok=v!=-1
    if np.issubdtype(v.dtype, np.floating): ok &= ~np.isnan(v)
    return ok

def _chain_pairs(labels, t, ept, valid=None):
    n=len(labels); pos=np.arange(n); gl=np.asarray(labels); tv=np.asarray(t)
    if valid is not None: pos=pos[valid]; gl=gl[valid]; tv=tv[valid]
    if len(pos)==0: z=np.array([],np.int64); return z,z
    order=np.argsort(tv,kind="mergesort")
    g=pd.DataFrame({"node":pos[order],"g":gl[order]}); sl,dl=[],[]
    for _,grp in g.groupby("g",sort=False):
        idx=grp["node"].to_numpy(); Lg=len(idx)
        for j in range(1,ept):
            if j>=Lg: break
            sl.append(idx[:Lg-j]); dl.append(idx[j:])
    if sl: return np.concatenate(sl), np.concatenate(dl)
    z=np.array([],np.int64); return z,z

def build_edges():
    parts=[]   # (src,dst,rel_idx)  rel: 0 uid,1 c1a1,2 c1a1e,3 dev,4 sim
    for ridx,col in enumerate(IDENTITY_COLS):
        s,d=_chain_pairs(id_vals[col], times, EDGE_PER_TRANS, valid=_valid_id(id_vals[col]))
        if len(s): parts.append((s,d,ridx)); print(f"  {col}: {len(s):,} edges")
    if SIM_BACKEND=="gpu":
        s,d = knn_gpu(X_norm, times, uids, K_SIM, device, batch=SIM_BATCH)
    else:
        I = knn_faiss(X_norm, K_SIM); rows=np.repeat(np.arange(N),K_SIM); cols=I[:,1:].reshape(-1)
        keep = times[cols] < times[rows]                     # causal filter post-hoc
        keep &= (pd.factorize(uids)[0][cols] != pd.factorize(uids)[0][rows])
        s,d = cols[keep], rows[keep]
    if len(s): parts.append((s,d,4)); print(f"  similarity(kNN k={K_SIM}): {len(s):,} edges")
    src=np.concatenate([p[0] for p in parts]); dst=np.concatenate([p[1] for p in parts])
    rel=np.concatenate([np.full(len(p[0]),p[2],np.int8) for p in parts])
    # coalesce duplicate (src,dst): OR relation flags
    e=pd.DataFrame({"s":src,"d":dst})
    for j in range(5): e[f"r{j}"]=(rel==j).astype(np.float32)
    agg=e.groupby(["s","d"],sort=False).max().reset_index()
    s2=agg["s"].to_numpy(); d2=agg["d"].to_numpy()
    relflags=agg[[f"r{j}" for j in range(5)]].to_numpy(np.float32)
    relcount=relflags.sum(1,keepdims=True)
    dt=((times[d2]-times[s2]).astype(np.float32))/86400.0
    rec=np.stack([np.exp(-dt/float(tt)) for tt in DECAY_TAUS],axis=1)
    cols_list=[rec, relflags, np.zeros((len(s2),1),np.float32), relcount]   # ..., is_self=0, rel_count
    if USE_COSINE:
        cos=np.empty((len(s2),1),np.float32); B=1_000_000
        for i in range(0,len(s2),B):
            cos[i:i+B,0]=(X_norm[s2[i:i+B]]*X_norm[d2[i:i+B]]).sum(1)
        cols_list.append(cos)
    ea=np.concatenate(cols_list,axis=1).astype(np.float32)
    ei=torch.tensor(np.stack([s2,d2]),dtype=torch.long); ea=torch.tensor(ea)
    # self-loops: rec=1, flags=0, is_self=1, rel_count=0, cosine=1
    D=ea.shape[1]; self_ea=torch.zeros((N,D)); self_ea[:,0:3]=1.0; self_ea[:,8]=1.0
    if USE_COSINE: self_ea[:,10]=1.0
    loops=torch.arange(N); ei=torch.cat([ei,torch.stack([loops,loops])],1); ea=torch.cat([ea,self_ea],0)
    print(f"  TOTAL: {ei.shape[1]:,} edges (incl self-loops), edge_dim={ea.shape[1]}")
    return ei, ea

# cache the built graph so seed re-runs / tweaks skip the kNN+coalesce rebuild
_tag = f"k{K_SIM}_ept{EDGE_PER_TRANS}_cos{int(USE_COSINE)}_{SIM_BACKEND}"
_ei_p, _ea_p = OUT/f"edge_index_{_tag}.npy", OUT/f"edge_attr_{_tag}.npy"
if _ei_p.exists() and _ea_p.exists():
    edge_index = torch.from_numpy(np.load(_ei_p)); edge_attr = torch.from_numpy(np.load(_ea_p))
    print(f"loaded cached graph: {edge_index.shape[1]:,} edges, edge_dim={edge_attr.shape[1]}")
else:
    t0=time.time(); edge_index, edge_attr = build_edges(); print(f"edges built in {time.time()-t0:.0f}s")
    np.save(_ei_p, edge_index.numpy()); np.save(_ea_p, edge_attr.numpy())
EDGE_DIM = edge_attr.shape[1]


  uid: 1,791,803 edges
  card1_addr1: 5,229,254 edges
  card1_addr1_P_emaildomain: 4,099,004 edges
  DeviceInfo: 1,233,979 edges
  similarity(kNN k=15): 8,857,980 edges
  TOTAL: 18,645,286 edges (incl self-loops), edge_dim=11
edges built in 209s


In [21]:
# ===== Edge-aware GTAN (edge_dim from edge_attr) =====
def _make_y_input(y, known_idx, n_classes):
    yi=torch.full((y.shape[0],), n_classes, dtype=torch.long); yi[known_idx]=y[known_idx]; return yi

class GTANEdge(nn.Module):
    def __init__(self, in_feats, hidden=32, heads=4, layers=2, n_classes=2, drop=0.2, edge_dim=11):
        super().__init__(); width=hidden*heads; self.n_classes=n_classes
        self.label_emb=nn.Embedding(n_classes+1, in_feats, padding_idx=n_classes)
        self.feat_lin=nn.Linear(in_feats,width); self.label_lin=nn.Linear(in_feats,width)
        self.label_proc=nn.Sequential(nn.BatchNorm1d(width),nn.PReLU(),nn.Dropout(drop),nn.Linear(width,in_feats))
        self.input_drop=nn.Dropout(drop)
        self.convs=nn.ModuleList(); self.norms=nn.ModuleList(); dim=in_feats
        for _ in range(layers):
            self.convs.append(TransformerConv(dim,hidden,heads=heads,concat=True,beta=True,dropout=drop,edge_dim=edge_dim))
            self.norms.append(nn.LayerNorm(width)); dim=width
        self.act=nn.PReLU(); self.drop=nn.Dropout(drop); self.emb_dim=dim
        self.head=nn.Sequential(nn.Linear(dim,dim),nn.BatchNorm1d(dim),nn.PReLU(),nn.Dropout(drop),nn.Linear(dim,n_classes))
    def forward(self,x,ei,yin,ea=None):
        le=self.input_drop(self.label_emb(yin)); h=x+self.label_proc(self.feat_lin(x)+self.label_lin(le))
        for conv,norm in zip(self.convs,self.norms): h=self.drop(self.act(norm(conv(h,ei,edge_attr=ea))))
        return self.head(h), h


In [22]:
# ===== Train (reference revealed, query supervised), score val =====
def run_gtan(seed):
    torch.manual_seed(seed); np.random.seed(seed)
    x=torch.tensor(X_scaled); y=torch.tensor(y_all,dtype=torch.long)
    yik=_make_y_input(y, torch.tensor(ref_idx), 2)            # reference labels revealed, rest=2
    npos=float((y[torch.tensor(query_idx)]==1).sum().clamp(min=1)); nneg=float((y[torch.tensor(query_idx)]==0).sum().clamp(min=1))
    weight=torch.tensor([1.0,float(np.sqrt(nneg/npos))],device=device)
    model=GTANEdge(x.shape[1],GTAN_HIDDEN,GTAN_HEADS,GTAN_LAYERS,2,GTAN_DROP,EDGE_DIM).to(device)
    opt=torch.optim.Adam(model.parameters(),lr=GTAN_LR,weight_decay=GTAN_WD)
    data=Data(x=x.float(), edge_index=edge_index, y=y); data.edge_attr=edge_attr.float(); data.yik=yik
    tl=NeighborLoader(data,num_neighbors=list(GTAN_NEIGH),input_nodes=torch.tensor(query_idx),batch_size=GTAN_BATCH,shuffle=True)
    vl=NeighborLoader(data,num_neighbors=list(GTAN_NEIGH),input_nodes=torch.tensor(val_idx),batch_size=GTAN_BATCH,shuffle=False)
    best,bestp,bad=-1.0,None,0
    for ep in range(1,GTAN_EPOCHS+1):
        model.train(); tot=nb=0
        for b in tl:
            bs=b.batch_size; b=b.to(device)
            opt.zero_grad(); logits,_=model(b.x,b.edge_index,b.yik,b.edge_attr)   # query seeds already =2 in yik
            loss=F.cross_entropy(logits[:bs], b.y[:bs], weight=weight); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),2.0); opt.step(); tot+=float(loss); nb+=1
        model.eval(); vp=np.zeros(N,np.float32)
        with torch.no_grad():
            for b in vl:
                bs=b.batch_size; b=b.to(device)
                vp[b.n_id[:bs].cpu().numpy()]=F.softmax(model(b.x,b.edge_index,b.yik,b.edge_attr)[0][:bs],1)[:,1].cpu().numpy()
        au=roc_auc_score(y_all[val_idx], vp[val_idx])
        print(f"   ep {ep:2d} loss={tot/max(nb,1):.4f} val_auc={au:.4f}")
        if au>best: best=au; bestp=vp[val_idx].copy(); bad=0
        else:
            bad+=1
            if bad>=EARLY_STOP_PATIENCE: print(f"   early stop at ep {ep}"); break
    print(f">>> seed {seed}: best val AUC={best:.4f}")
    del model; gc.collect()
    if device.type=="cuda": torch.cuda.empty_cache()
    return best, bestp

seed_aucs=[]; seed_preds=[]
for s in range(N_SEEDS):
    print(f"\n-- GTAN seed {SEED+s} --")
    a,p=run_gtan(SEED+s); seed_aucs.append(a); seed_preds.append(p)
avg=np.mean(seed_preds,axis=0)
print(f"\n=== GTAN-alone uid-disjoint ===")
print(f"per-seed val AUC: {[round(a,4) for a in seed_aucs]}")
print(f"seed-averaged val AUC = {roc_auc_score(y_all[val_idx], avg):.4f}")
print(f"seed-averaged val AP  = {average_precision_score(y_all[val_idx], avg):.4f}")



-- GTAN seed 42 --
   ep  1 loss=0.3480 val_auc=0.8596
   ep  2 loss=0.2945 val_auc=0.8650
   ep  3 loss=0.2820 val_auc=0.8684
   ep  4 loss=0.2715 val_auc=0.8666
   ep  5 loss=0.2626 val_auc=0.8669
   ep  6 loss=0.2564 val_auc=0.8584
   ep  7 loss=0.2496 val_auc=0.8567
   early stop at ep 7
>>> seed 42: best val AUC=0.8684

-- GTAN seed 43 --
   ep  1 loss=0.3374 val_auc=0.8554
   ep  2 loss=0.2929 val_auc=0.8708
   ep  3 loss=0.2796 val_auc=0.8753
   ep  4 loss=0.2697 val_auc=0.8756
   ep  5 loss=0.2617 val_auc=0.8737
   ep  6 loss=0.2547 val_auc=0.8698
   ep  7 loss=0.2494 val_auc=0.8666
   ep  8 loss=0.2426 val_auc=0.8595
   early stop at ep 8
>>> seed 43: best val AUC=0.8756

-- GTAN seed 44 --
   ep  1 loss=0.3436 val_auc=0.8587
   ep  2 loss=0.2945 val_auc=0.8659
   ep  3 loss=0.2803 val_auc=0.8737
   ep  4 loss=0.2719 val_auc=0.8744
   ep  5 loss=0.2623 val_auc=0.8767
   ep  6 loss=0.2551 val_auc=0.8704
   ep  7 loss=0.2495 val_auc=0.8717
   ep  8 loss=0.2425 val_auc=0.8646
  